<a href="https://colab.research.google.com/github/weagan/Speculative-Decoding/blob/main/target_drafter_auto_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dual‑GPU Speculative Decoding

Target model: **meta-llama/Llama-3.1-8B** NF4 4-bit, spread across both GPUs via `device_map='auto'`
Draft model: **meta-llama/Llama-3.2-1B** NF4 4-bit, pinned to GPU 1

Both models share the Llama-3 128k vocabulary → logit-level verification.

Memory layout on two 16 GB T4s (~29 GB usable after driver overhead):
- Target 8B @ NF4 ≈ 5 GB, split ~4 GB on GPU 0 + ~1 GB on GPU 1 by accelerate
- Draft  1B @ NF4 ≈ 0.7 GB on GPU 1

> Explain why transformers use self-attention.


In [ ]:
!pip install -q transformers accelerate sentencepiece safetensors bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inc

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

assert torch.cuda.device_count() >= 2, 'Need 2 GPUs'

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Missing Kaggle secret HF_TOKEN')

target_name = 'meta-llama/Llama-3.1-8B'
draft_name  = 'meta-llama/Llama-3.2-1B'

tokenizer = AutoTokenizer.from_pretrained(target_name, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Draft: NF4 4-bit, pinned entirely to GPU 1 (~0.7 GB)
draft_model = AutoModelForCausalLM.from_pretrained(
    draft_name,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map={'': 1},
    low_cpu_mem_usage=True,
)
draft_model.eval()
print('Draft device:', next(draft_model.parameters()).device)

# Target: NF4 4-bit, device_map='auto' lets accelerate split layers across
# both GPUs so the total ~5 GB never has to fit on one card.
# We tell accelerate how much free memory to leave on each GPU so the
# draft model and activations are not evicted.
max_memory = {
    0: '13GiB',   # GPU 0: give target almost everything
    1: '7GiB',    # GPU 1: shared with draft (~0.7 GB) + activations
    'cpu': '20GiB',
}

target_model = AutoModelForCausalLM.from_pretrained(
    target_name,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map='auto',
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)
target_model.eval()
print('Target device map:', target_model.hf_device_map)


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Target device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [ ]:
from concurrent.futures import ThreadPoolExecutor

# One persistent executor — two workers, one per GPU.
# Keeping it alive avoids thread-spawn overhead each round.
_executor = ThreadPoolExecutor(max_workers=2)


@torch.inference_mode()
def generate(model, prompt, max_new_tokens=128):
    """Plain autoregressive generation — baseline timing only."""
    first_device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors='pt').to(first_device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.3,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def strip_prompt(prompt, full):
    return full[len(prompt):] if full.startswith(prompt) else full


TARGET_INPUT_DEVICE = target_model.model.embed_tokens.weight.device
print('Target input device:', TARGET_INPUT_DEVICE)


@torch.inference_mode()
def draft_generate_ids(input_ids, draft_step):
    """Draft autoregressive generation on GPU 1. Returns 1-D token ID tensor."""
    out = draft_model.generate(
        input_ids.unsqueeze(0).to('cuda:1'),
        max_new_tokens=draft_step,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.3,
    )
    return out[0]  # (prompt_len + draft_step,)


@torch.inference_mode()
def target_logits_for_draft(input_ids, draft_ids):
    """One forward pass of the target. Returns (n_draft, V) log-probs."""
    seq = draft_ids.unsqueeze(0).to(TARGET_INPUT_DEVICE)
    logits = target_model(seq).logits
    prompt_len = input_ids.shape[0]
    n_draft = draft_ids.shape[0] - prompt_len
    relevant = logits[0, prompt_len - 1 : prompt_len - 1 + n_draft]
    return torch.log_softmax(relevant.float(), dim=-1)


@torch.inference_mode()
def draft_logits_for_sequence(input_ids, draft_ids):
    """One forward pass of the draft. Returns (n_draft, V) log-probs."""
    seq = draft_ids.unsqueeze(0).to('cuda:1')
    logits = draft_model(seq).logits
    prompt_len = input_ids.shape[0]
    n_draft = draft_ids.shape[0] - prompt_len
    relevant = logits[0, prompt_len - 1 : prompt_len - 1 + n_draft]
    return torch.log_softmax(relevant.float(), dim=-1)


Target input device: cuda:0


## Test run on each model

In [ ]:
import time

prompt = 'Explain why transformers use self-attention.'

print('=== Target model ===')
start = time.time()
t = generate(target_model, prompt, 200)
elapsed = time.time() - start
t_tokens = len(tokenizer.encode(strip_prompt(prompt, t)))
print(strip_prompt(prompt, t))
print(f'Target model: {elapsed:.2f}s  {t_tokens/elapsed:.2f} tok/s')

print('\n=== Draft model ===')
start = time.time()
d = generate(draft_model, prompt, 200)
elapsed = time.time() - start
d_tokens = len(tokenizer.encode(strip_prompt(prompt, d)))
print(strip_prompt(prompt, d))
print(f'Draft model:  {elapsed:.2f}s  {d_tokens/elapsed:.2f} tok/s')


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Target model ===
 What are the advantages of using it?
Self attention is a mechanism that allows an input to attend on different positions in its own sequence, and then combine this information with itself at each position.
The main advantage here would be allowing for more complex interactions between words than just left-to-right or right-to-left dependencies (which can also be modeled by recurrent networks). For example:
“the man” -> “man”
“The dog chased his tail.” -> “chased”, “his”.
This type of interaction could not easily happen without some form of context awareness like what we see from transformer models today!
What does Transformer mean? How do you explain Transformers as they relate back into natural language processing tasks such as machine translation
A transformer model uses multiple layers where one layer takes care only about encoding while another will take over decoding task after being trained together during training process so both encoder-decoder pair works 

## Speculative decoding — logit-level verification

In [ ]:
import torch

@torch.inference_mode()
def speculative_decode(prompt, max_new_tokens=256, draft_step=8):
    """
    Logit-level speculative decoding with parallel target/draft verification.

    Each round:
      1. Draft generates `draft_step` tokens autoregressively on GPU 1.
      2+3. Target (GPU 0/1 split) and draft (GPU 1) each do ONE forward pass
           over the draft sequence — launched in parallel via ThreadPoolExecutor
           so both GPUs are busy simultaneously.
      4. Accept token t_i with prob min(1, p_target(t_i) / p_draft(t_i)).
         On first rejection resample from max(0, p_target - p_draft).
      5. If all tokens accepted, sample one free bonus token from target.
    """
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids'][0]  # (L,)
    accepted_ids = input_ids.clone()  # kept on CPU; moved to GPU only for kernel calls
    total_new = 0
    round_num = 0

    while total_new < max_new_tokens:
        round_num += 1
        # ── 1. Draft generates candidates on GPU 1 ────────────────────
        draft_ids = draft_generate_ids(accepted_ids, draft_step)
        n_draft = draft_ids.shape[0] - accepted_ids.shape[0]
        if n_draft == 0:
            break

        # ── 2 & 3. Parallel forward passes on both GPUs ───────────────
        # Submit both to the thread pool; CUDA streams on each GPU run
        # concurrently while Python waits on both futures.
        fut_target = _executor.submit(target_logits_for_draft, accepted_ids, draft_ids)
        fut_draft  = _executor.submit(draft_logits_for_sequence, accepted_ids, draft_ids)
        log_p_target = fut_target.result()  # (n_draft, V) on TARGET_INPUT_DEVICE
        log_p_draft  = fut_draft.result()   # (n_draft, V) on cuda:1

        p_target = log_p_target.exp().to(TARGET_INPUT_DEVICE)
        p_draft  = log_p_draft.exp().to(TARGET_INPUT_DEVICE)

        # ── 4. Token-by-token acceptance ──────────────────────────────
        new_token_ids = draft_ids[accepted_ids.shape[0]:].to(TARGET_INPUT_DEVICE)
        n_accepted = 0

        for i in range(n_draft):
            t = new_token_ids[i].item()
            ratio = p_target[i, t] / (p_draft[i, t] + 1e-9)
            u = torch.rand(1, device=TARGET_INPUT_DEVICE).item()

            if u < ratio:  # accept
                n_accepted += 1
                print(f"  Round {round_num}: Accepted token {i+1}/{n_draft}")
                if t == tokenizer.eos_token_id:
                    accepted_ids = torch.cat([
                        accepted_ids,
                        new_token_ids[:n_accepted].cpu()
                    ])
                    total_new += n_accepted
                    print(f"  Round {round_num}: Total accepted in round: {n_accepted}")
                    return tokenizer.decode(
                        accepted_ids[input_ids.shape[0]:], skip_special_tokens=True
                    )
            else:          # reject — resample from residual
                print(f"  Round {round_num}: Rejected token {i+1}/{n_draft}")
                residual = torch.clamp(p_target[i] - p_draft[i], min=0.0)
                s = residual.sum()
                if s > 0:
                    residual /= s
                    t_new = torch.multinomial(residual, 1).item()
                else:
                    t_new = torch.argmax(p_target[i]).item()
                new_token_ids[i] = t_new
                n_accepted += 1
                break
        print(f"  Round {round_num}: Total accepted in round: {n_accepted}")

        accepted_ids = torch.cat([
            accepted_ids,
            new_token_ids[:n_accepted].cpu()
        ])
        total_new += n_accepted

        # ── 5. Bonus token when all drafts accepted ────────────────────
        if n_accepted == n_draft:
            print(f"  Round {round_num}: All draft tokens accepted, sampling bonus token.")
            bonus_seq = accepted_ids.unsqueeze(0).to(TARGET_INPUT_DEVICE)
            bonus_logits = target_model(bonus_seq).logits[0, -1]
            bonus_token = torch.argmax(bonus_logits).item()
            if bonus_token == tokenizer.eos_token_id:
                print(f"  Round {round_num}: Bonus token is EOS, stopping.")
                break
            accepted_ids = torch.cat([
                accepted_ids,
                torch.tensor([bonus_token])
            ])
            total_new += 1
            print(f"  Round {round_num}: Bonus token {bonus_token} added. Total new: {total_new}")

    return tokenizer.decode(accepted_ids[input_ids.shape[0]:], skip_special_tokens=True)

In [ ]:
import time

start = time.time()
result = speculative_decode(prompt, max_new_tokens=256, draft_step=8)
elapsed = time.time() - start
result_tokens = len(tokenizer.encode(result))
print(f'Speculative decoding: {elapsed:.2f}s  {result_tokens/elapsed:.2f} tok/s')
print()
print(prompt + result)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  Round 1: Rejected token 1/8
  Round 1: Total accepted in round: 1
  Round 2: Rejected token 1/8
  Round 2: Total accepted in round: 1
  Round 3: Accepted token 1/8
  Round 3: Accepted token 2/8
  Round 3: Rejected token 3/8
  Round 3: Total accepted in round: 3
  Round 4: Accepted token 1/8
  Round 4: Accepted token 2/8
  Round 4: Rejected token 3/8
  Round 4: Total accepted in round: 3
  Round 5: Rejected token 1/8
  Round 5: Total accepted in round: 1
  Round 6: Accepted token 1/8
  Round 6: Accepted token 2/8
  Round 6: Rejected token 3/8
  Round 6: Total accepted in round: 3
  Round 7: Accepted token 1/8
  Round 7: Accepted token 2/8
  Round 7: Accepted token 3/8
  Round 7: Accepted token 4/8
  Round 7: Rejected token 5/8
  Round 7: Total accepted in round: 5
  Round 8: Accepted token 1/8
  Round 8: Accepted token 2/8
  Round 8: Accepted token 3/8
  Round 8: Rejected token 4/8
  Round 8: Total accepted in round: 4
  Round 9: Accepted token 1/2
  Round 9: Accepted token 2/2
  Roun